# Multimodal Financial Market Analysis — BBCA (NeuralProphet + MIDAS)
**Authors**: Wesley Coa, Geoffrey Gohtama, Wilbert — Bina Nusantara University

Implements the methodology described in the pre-thesis report (Section III):
MIDAS-aggregated weekly price features (Almon lag polynomial), R-MIDAS-projected
weekly fundamental features, and two NeuralProphet models (Price-Only vs Hybrid)
evaluated with MAE, RMSE, and directional accuracy.

**Install:**
```
pip install neuralprophet pdfplumber scikit-learn matplotlib pandas numpy
```
**Files needed (same folder as notebook):**
- `BBCA_2015_2025_Combined.csv`
- `monthly_reports/BBCA_YYYY_MM.pdf`

> **Assumption flagged for review:** Section III-B's "Train-Test Splitting" step
> says "the final 120 trading days are used for testing," but the paragraphs
> right above it aggregate both price and fundamentals onto a **weekly** grid
> before modeling. This notebook therefore holds out the final **120 weekly**
> observations (~2.3 years) as the test set, matching the frequency the rest of
> the methodology operates on. If the report actually meant 120 *daily* trading
> days re-expressed as roughly 24 weeks, change `TEST_SIZE` in Cell 2.

In [ ]:
# ============================================================
#  CELL 1 — IMPORTS
# ============================================================
import os, re, glob, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from neuralprophet import NeuralProphet, set_log_level
    set_log_level('ERROR')
except ImportError:
    raise ImportError('Run: pip install neuralprophet')

try:
    import pdfplumber
except ImportError:
    raise ImportError('Run: pip install pdfplumber')

# Compat shim: PyTorch >=2.6 flipped torch.load's default to weights_only=True,
# which breaks PyTorch Lightning's internal LR-finder checkpoint restore under
# neuralprophet 0.9.0 (it unpickles neuralprophet's own Config* classes, which
# aren't on torch's safe-globals allowlist). Checkpoints here are always ones
# we just wrote ourselves, so restoring with weights_only=False is safe.
import torch
_torch_load = torch.load
def _torch_load_compat(*args, **kwargs):
    kwargs['weights_only'] = False
    return _torch_load(*args, **kwargs)
torch.load = _torch_load_compat

print('All imports OK')

All imports OK


In [ ]:
# ============================================================
#  CELL 2 — CONFIGURATION
# ============================================================
PRICE_PATH   = 'BBCA_2015_2025_Combined.csv'
PDF_DIR      = 'monthly_reports'
PARSED_CACHE = 'BBCA_monthly_parsed.csv'

TEST_SIZE    = 120     # final 120 WEEKLY observations held out (see note above)
PUB_LAG_DAYS = 45      # BCA publishes ~4-6 weeks after period end (no look-ahead)

# MIDAS -- Almon lag polynomial (degree-2) weighting.
# Weights decay smoothly from the most recent observation (lag 0) backward;
# theta values are fixed defaults rather than MLE-fitted, since the report
# does not specify a fitting procedure for them.
ALMON_DEGREE         = 2
ALMON_THETA_PX       = (0.35, -0.05)   # weekly PRICE aggregation (daily -> weekly)
ALMON_THETA_FUND     = (0.55, -0.08)   # R-MIDAS fundamentals (monthly -> weekly)
FUND_LOOKBACK_MONTHS = 3               # how many past monthly reports feed each week

# NeuralProphet hyperparameters -- shared by both models (Section III-C)
N_LAGS      = 8    # weekly autoregressive window (~2 months of momentum)
N_FORECASTS = 1    # predicts the next weekly closing price
EPOCHS      = 50

# Fundamental indicators used, exactly as named in Section III-A
FUNDAMENTAL_COLS = [
    'net_income',
    'revenue',
    'net_profit_margin',
    'ni_growth_yoy',
    'rev_growth_yoy',
]

print('Config OK')

Config OK


In [ ]:
# ============================================================
#  CELL 3 — PRICE DATA LOADER
# ============================================================
def load_price_data(path):
    df = pd.read_csv(path)
    for fmt in ('%m/%d/%Y', '%Y-%m-%d', '%d/%m/%Y'):
        try:
            df['Date'] = pd.to_datetime(df['Date'], format=fmt)
            break
        except (ValueError, TypeError):
            continue
    else:
        df['Date'] = pd.to_datetime(df['Date'], infer_datetime_format=True)

    df = (df.sort_values('Date')
            .reset_index(drop=True)
            .rename(columns={'Date': 'ds', 'Close': 'y'})[['ds', 'y']])
    print(f'[Price]  {df["ds"].min().date()} -> {df["ds"].max().date()}'
          f'  |  {len(df):,} trading days')
    return df

In [ ]:
# ============================================================
#  CELL 4 — YTD CUMULATIVE -> STANDALONE MONTHLY
#
#  OJK monthly reports are titled "For Periods Ended YYYY-MM-DD"
#  meaning income-statement figures are CUMULATIVE year-to-date.
#  e.g. April 2015 report = Jan+Feb+Mar+Apr combined, not April alone.
#  Balance-sheet items (total_assets, total_loans) are snapshots -- no fix.
# ============================================================
_YTD_COLS = ['net_profit', 'interest_income', 'interest_expense', 'personnel_expense']


def ytd_to_monthly(series, dates):
    result = series.copy().astype(float)
    for yr in dates.dt.year.unique():
        mask   = dates.dt.year == yr
        idx    = series.index[mask]
        vals   = series.loc[mask].values.astype(float)
        months = dates.loc[mask].dt.month.values
        order  = np.argsort(months)
        idx, vals, months = idx[order], vals[order], months[order]
        standalone = np.empty_like(vals)
        standalone[0] = vals[0]   # January = standalone (1-month YTD)
        for i in range(1, len(vals)):
            diff = vals[i] - vals[i - 1]
            # If negative (data anomaly), use average-month estimate
            standalone[i] = diff if diff >= 0 else vals[i] / months[i]
        result.loc[idx] = standalone
    return result

In [ ]:
# ============================================================
#  CELL 5 — PDF REGEX PATTERNS
#
#  Two OJK layouts exist:
#
#  A) MONTHLY (Jan-Nov): single column, one date.
#     "1. Interest income 14,308,684"
#     "NET PROFIT (LOSS) 5,230,451"
#
#  B) ANNUAL DECEMBER: 4-column layout.
#     "1. Interest income  62,022,745  60,508,105  65,875,355  64,351,925"
#     We extract the FIRST number = Individual (Bank-only) current year.
#     Also has a Financial Ratios page with pre-computed NIM, LDR, ROA, CAR.
# ============================================================

# Monthly + annual income/balance sheet patterns
_PAT_MONTHLY = [
    ('interest_income', [
        r'1\.\s*Interest income\s+([\d,]+)',
        r'1\.\s*Pendapatan Bunga\s+([\d,]+)',
        r'Interest income[^\n]{0,40}([\d,\.]{6,})',
    ]),
    ('interest_expense', [
        r'2\.\s*Interest expenses?\s+([\d,]+)',
        r'2\.\s*Beban Bunga\s+([\d,]+)',
    ]),
    ('net_profit', [
        r'NET PROFIT \(LOSS\) AFTER TAX\s+([\d,]+)',
        r'NET PROFIT \(LOSS\)\s+([\d,]+)',
        r'LABA \(RUGI\) BERSIH\s+([\d,]+)',
        r'NET PROFIT\s+([\d,]{6,})',
    ]),
    ('personnel_expense', [
        r'(?:j\.\s*)?Personnel expenses\s+([\d,]+)',
        r'Beban Tenaga Kerja\s+([\d,]+)',
    ]),
    ('total_assets', [
        r'TOTAL ASSETS\s+([\d,]+)',
        r'TOTAL ASET\s+([\d,]+)',
    ]),
    ('total_loans', [
        r'9\.\s*Loans and financing\s+([\d,]+)',
        r'9\.\s*Loans\s+([\d,]+)',
        r'9\.\s*Kredit\s+([\d,]+)',
    ]),
    ('current_accounts', [
        r'1\.\s*Current account\s+([\d,]+)',
        r'1\.\s*Giro\s+([\d,]+)',
    ]),
    ('savings_accounts', [
        r'2\.\s*Saving account\s+([\d,]+)',
        r'2\.\s*Tabungan\s+([\d,]+)',
    ]),
    ('time_deposits', [
        r'3\.\s*Time deposit\s+([\d,]+)',
        r'3\.\s*Deposito\s+([\d,]+)',
    ]),
]

# December annual report -- pre-computed financial ratios page (unused by the
# current FUNDAMENTAL_COLS, kept so Appendix A / future extensions still work)
_PAT_RATIOS = [
    ('nim_reported',  [r'Net Interest Margin\s*\(NIM\)\s+([\d\.]+)%']),
    ('ldr_reported',  [r'Loan to Deposit Ratio\s*\(LDR\)\s+([\d\.]+)%']),
    ('roa_reported',  [r'Return on Asset\s*\(ROA\)\s+([\d\.]+)%']),
    ('roe_reported',  [r'Return on Equity\s*\(ROE\)\s+([\d\.]+)%']),
    ('car_reported',  [
        r'CAR Ratio\s*\(%\)\s+([\d\.]+)%',
        r'Capital Adequacy Ratio\s*\(CAR\)\s+([\d\.]+)%',
    ]),
    ('npl_gross',     [r'Gross NPL\s+([\d\.]+)%']),
    ('bopo_reported', [r'Operating Expenses to Operating Income\s*\(BOPO\)\s+([\d\.]+)%']),
]


def _is_annual_format(text):
    """December annual reports have INDIVIDUAL and CONSOLIDATED columns side-by-side."""
    return bool(
        re.search(r'INDIVIDUAL\s+CONSOLIDATED', text, re.IGNORECASE) or
        re.search(r'Dec 31,\s*\d{4}\s+Dec 31,\s*\d{4}', text)
    )


def _clean_number(raw):
    raw = raw.strip().replace(' ', '').replace(',', '')
    try:
        return float(raw)
    except ValueError:
        return np.nan


def _extract_fields(text, patterns):
    result = {}
    for col, pats in patterns:
        found = np.nan
        for pat in pats:
            m = re.search(pat, text, re.IGNORECASE)
            if m:
                found = _clean_number(m.group(1))
                if not np.isnan(found):
                    break
        result[col] = found
    return result


def _parse_single_pdf(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            full_text = '\n'.join(p.extract_text() or '' for p in pdf.pages)
    except Exception as e:
        print(f'  [WARN] Cannot open {os.path.basename(pdf_path)}: {e}')
        return None

    annual = _is_annual_format(full_text)
    values = _extract_fields(full_text, _PAT_MONTHLY)

    if annual:
        ratios = _extract_fields(full_text, _PAT_RATIOS)
        values.update(ratios)
        values['is_annual'] = 1.0
    else:
        values['is_annual'] = 0.0

    missing = [k for k in ('interest_income', 'net_profit')
               if np.isnan(values.get(k, np.nan))]
    if missing:
        print(f'  [WARN] {missing} not found in {os.path.basename(pdf_path)}')
    return values

print('PDF patterns OK')

PDF patterns OK


In [ ]:
# ============================================================
#  CELL 6 — LOAD & CLEAN MONTHLY PDF DATA
# ============================================================
def load_monthly_pdf_data(pdf_dir, cache_path):

    # 1. Load cache
    if os.path.exists(cache_path):
        cached = pd.read_csv(cache_path, parse_dates=['ds'])
        cached_months = set(cached['ds'].dt.to_period('M').astype(str))
        print(f'[Monthly]  Cache: {len(cached)} months loaded.')
    else:
        cached = pd.DataFrame()
        cached_months = set()

    # 2. Parse new PDFs
    pdf_files = sorted(glob.glob(os.path.join(pdf_dir, 'BBCA_????_??.pdf')))
    if not pdf_files:
        raise FileNotFoundError(
            f'No PDFs found in "{pdf_dir}". Expected: BBCA_YYYY_MM.pdf')

    new_rows = []
    for pdf_path in pdf_files:
        fname = os.path.basename(pdf_path)
        m = re.match(r'BBCA_(\d{4})_(\d{2})\.pdf', fname)
        if not m:
            continue
        year, month = int(m.group(1)), int(m.group(2))
        period_end  = pd.Timestamp(year, month, 1) + pd.offsets.MonthEnd(0)
        period_key  = period_end.to_period('M').strftime('%Y-%m')
        if period_key in cached_months:
            continue

        print(f'  [Parse] {fname} ...', end=' ')
        values = _parse_single_pdf(pdf_path)
        if values is None:
            continue

        row = {'ds': period_end}
        for col, val in values.items():
            if col == 'is_annual':
                row[col] = val
            else:
                row[col] = val / 1_000 if not np.isnan(val) else np.nan  # M -> B IDR
        new_rows.append(row)
        ni  = row.get('net_profit',      float('nan'))
        rev = row.get('interest_income', float('nan'))
        fmt = 'ANNUAL' if row.get('is_annual') else 'monthly'
        print(f'[{fmt}]  NI={ni:.0f}B  Rev={rev:.0f}B')

    # 3. Merge & save
    if new_rows:
        new_df  = pd.DataFrame(new_rows)
        all_raw = (pd.concat([cached, new_df], ignore_index=True)
                   if not cached.empty else new_df)
        all_raw = all_raw.sort_values('ds').reset_index(drop=True)
        all_raw.to_csv(cache_path, index=False)
        print(f'[Monthly]  {len(new_rows)} new months saved.')
    else:
        all_raw = cached.copy()

    raw = all_raw.sort_values('ds').reset_index(drop=True)

    # 4. YTD -> standalone monthly for income-statement columns
    print('[Monthly]  Converting YTD cumulative -> standalone monthly ...')
    for col in _YTD_COLS:
        if col in raw.columns:
            raw[col] = ytd_to_monthly(raw[col], raw['ds'])

    # 5. Rename to match Section III-A naming
    raw = raw.rename(columns={
        'net_profit':      'net_income',
        'interest_income': 'revenue',
    })

    # 6. Derived features (Section III-A: Net Income, Revenue, Net Profit
    #    Margin as levels; Net Income Growth Y-Y and Revenue Growth Y-Y)
    raw['net_profit_margin'] = (raw['net_income'] / raw['revenue']).clip(0, 1)
    raw['ni_growth_yoy']  = raw['net_income'].pct_change(12).fillna(0).clip(-1, 1)
    raw['rev_growth_yoy'] = raw['revenue'].pct_change(12).fillna(0).clip(-1, 1)

    # 7. Select final columns -- exactly the indicators named in Section III-A
    keep = ['ds'] + FUNDAMENTAL_COLS
    fund = raw[[c for c in keep if c in raw.columns]]
    fund = fund.dropna(subset=['net_profit_margin']).reset_index(drop=True)
    feat_cols = [c for c in fund.columns if c != 'ds']
    print(f'[Monthly]  {len(fund)} months ready  '
          f'({fund["ds"].min().date()} -> {fund["ds"].max().date()})')
    print(f'[Monthly]  Features: {feat_cols}')
    return fund

## Cell 7 — MIDAS: High-Frequency Aggregation & R-MIDAS Low-Frequency Disaggregation

In [ ]:
# ============================================================
#  CELL 7a -- MIDAS: HIGH-FREQUENCY AGGREGATION (daily -> weekly price)
#
#  Section III-B: "For a given week, the daily prices are aggregated
#  using an Almon lag polynomial to assign weights, forming the
#  weekly price feature."
#
#  Implementation: within each ISO week, the most recent trading day
#  gets lag index 0, the day before it lag 1, and so on. An Almon
#  (exponential) lag polynomial turns those lag indices into weights
#  that decay smoothly, and the weekly price is their weighted
#  average -- this keeps the day closest to the week's end most
#  influential while still letting earlier days in the week
#  contribute, rather than simply taking the Friday close.
# ============================================================
def almon_weights(n_lags, theta):
    """theta: tuple of Almon polynomial coefficients (theta1, theta2, ...)."""
    lags = np.arange(n_lags)
    exponent = sum(t * (lags ** (i + 1)) for i, t in enumerate(theta))
    w = np.exp(exponent)
    return w / w.sum()


def midas_weekly_price(df_price, theta=ALMON_THETA_PX):
    """
    Aggregate daily closing prices into one Almon-weighted value per
    ISO week. The week's label (`ds`) is set to that week's LAST
    trading day, so the weekly observation is only ever built from
    days up to and including that date (no look-ahead).
    """
    d = df_price.copy().sort_values('ds').reset_index(drop=True)
    iso = d['ds'].dt.isocalendar()
    d['iso_year'] = iso['year']
    d['iso_week'] = iso['week']

    rows = []
    for (iso_year, iso_week), grp in d.groupby(['iso_year', 'iso_week'], sort=False):
        grp = grp.sort_values('ds')
        prices = grp['y'].values[::-1]           # most recent day first (lag 0)
        w = almon_weights(len(prices), theta)
        weekly_price = float(np.dot(prices, w))
        week_monday = pd.to_datetime(
            f'{iso_year}-W{int(iso_week):02d}-1', format='%G-W%V-%u')
        rows.append({'ds': week_monday, 'y': weekly_price})

    weekly = pd.DataFrame(rows).sort_values('ds').reset_index(drop=True)
    gaps = weekly['ds'].diff().dropna()
    n_gaps = (gaps != pd.Timedelta(days=7)).sum()
    if n_gaps:
        print(f'[MIDAS]  WARNING: {n_gaps} week(s) are not exactly 7 days apart '
              f'(likely a fully non-trading ISO week, e.g. a holiday week).')
    print(f'[MIDAS]  Weekly price series: {len(weekly)} weeks  '
          f'({weekly["ds"].min().date()} -> {weekly["ds"].max().date()})')
    return weekly

# ============================================================
#  CELL 7b -- R-MIDAS: LOW-FREQUENCY DISAGGREGATION (monthly -> weekly)
#
#  Section III-B: "Monthly financial report data are mapped onto a
#  weekly grid using R-MIDAS framework. For each weekly observation,
#  the model incorporates a weighted combination of the most recent
#  monthly observation."
#
#  A publication lag (PUB_LAG_DAYS) is applied first so a report is
#  only usable once it would actually have been public, preventing
#  look-ahead bias. Each weekly row then blends the last
#  FUND_LOOKBACK_MONTHS published reports with Almon weights that
#  favour the most recently published one.
# ============================================================
def rmidas_weekly_fundamentals(weekly_dates, fund_m, feat_cols,
                                lookback=FUND_LOOKBACK_MONTHS,
                                theta=ALMON_THETA_FUND,
                                pub_lag_days=PUB_LAG_DAYS):
    lagged = fund_m.copy()
    lagged['ds'] = lagged['ds'] + pd.DateOffset(days=pub_lag_days)
    lagged = lagged.sort_values('ds').reset_index(drop=True)
    w_full = almon_weights(lookback, theta)

    out_rows = []
    for wk in weekly_dates:
        avail = lagged[lagged['ds'] <= wk]
        if len(avail) == 0:
            continue
        recent = avail.iloc[-lookback:]           # up to `lookback` most-recent reports
        recent = recent.iloc[::-1]                 # most recent first (lag 0)
        weights = w_full[:len(recent)]
        weights = weights / weights.sum()           # renormalise for early weeks
        blended = {'ds': wk}
        for c in feat_cols:
            blended[c] = float(np.dot(recent[c].values, weights))
        out_rows.append(blended)

    weekly_fund = pd.DataFrame(out_rows)
    print(f'[R-MIDAS]  Weekly fundamentals: {len(weekly_fund)} weeks  '
          f'(publication lag = {pub_lag_days} days, lookback = {lookback} reports)')
    return weekly_fund

In [ ]:
# ============================================================
#  CELL 8 -- FEATURE NORMALIZATION (z-score, train-only stats)
#
#  Section III-B: "Each column of feature is standardized using
#  z-score normalization." Stats are computed on the TRAINING split
#  only to avoid look-ahead, then applied to the full series.
# ============================================================
def fit_normaliser(train_df, cols):
    stats = {}
    for col in cols:
        if col in train_df.columns:
            mu    = train_df[col].mean()
            sigma = train_df[col].std()
            stats[col] = (mu, sigma + 1e-9)
    return stats


def apply_normaliser(df, stats):
    df = df.copy()
    for col, (mu, sigma) in stats.items():
        if col in df.columns:
            df[col] = (df[col] - mu) / sigma
    return df


def invert_normaliser(values, stats, col):
    mu, sigma = stats[col]
    return values * sigma + mu

print('Normalisation helpers OK')

Normalisation helpers OK


## Cell 9 — NeuralProphet Model (Price-Only vs Hybrid)

In [ ]:
# ============================================================
#  CELL 9 -- NEURALPROPHET MODEL (Section III-C)
#
#  Two configurations, same hyperparameters, trained under the same
#  conditions:
#    1. Price-Only  -- NeuralProphet sees only the weekly MIDAS price.
#    2. Hybrid       -- NeuralProphet also sees the weekly R-MIDAS
#                        fundamental features as lagged regressors.
#
#  Since n_forecasts=1, evaluation is one-step-ahead: predict() is
#  called on the full (train+test) frame so each test-week prediction
#  uses the TRUE prior weeks' price/fundamentals as lag inputs, never
#  the model's own earlier predictions.
#
#  Other NeuralProphet settings (changepoints, learning rate, etc.)
#  are left at their library defaults, since the report only specifies
#  n_lags, n_forecasts, and epochs.
# ============================================================
def build_neuralprophet(fund_cols=None):
    m = NeuralProphet(
        n_lags=N_LAGS,
        n_forecasts=N_FORECASTS,
        epochs=EPOCHS,
        normalize='off',        # data is already z-scored in Cell 8
    )
    if fund_cols:
        for col in fund_cols:
            m.add_lagged_regressor(names=col, normalize='off')
    return m


def train_neuralprophet(df_weekly, fund_cols, cutoff, y_stats, name):
    """
    df_weekly must contain 'ds', normalized 'y', and (if fund_cols)
    the normalized fundamental columns. Returns a results dict with
    predictions on the ORIGINAL price scale plus error metrics.
    """
    cols = ['ds', 'y'] + (fund_cols or [])
    df = df_weekly[cols].dropna().reset_index(drop=True)
    train_df = df[df['ds'] < cutoff].reset_index(drop=True)

    print(f'\n  [{name}]  train={len(train_df):,} weeks  '
          f'test={(df["ds"] >= cutoff).sum():,} weeks  '
          f'features={len(cols) - 2}')

    m = build_neuralprophet(fund_cols)
    # 'W-MON' matches midas_weekly_price's week label (each ISO week's Monday);
    # pandas' plain 'W' alias means week-ending-Sunday, which misaligns every
    # row and forces NeuralProphet to treat almost the whole series as missing.
    m.fit(train_df, freq='W-MON')

    forecast = m.predict(df)
    forecast = forecast.dropna(subset=['yhat1']).reset_index(drop=True)

    test_fc = forecast[forecast['ds'] >= cutoff]
    y_true = invert_normaliser(test_fc['y'].values,     y_stats, 'y')
    y_pred = invert_normaliser(test_fc['yhat1'].values, y_stats, 'y')
    dates_te = test_fc['ds'].values

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mp   = y_true.mean()
    da   = np.mean(np.sign(np.diff(y_true)) == np.sign(np.diff(y_pred))) * 100

    metrics = dict(MAE=mae, RMSE=rmse,
                   MAE_pct=mae / mp * 100, RMSE_pct=rmse / mp * 100, Dir_Acc=da)

    print(f'     MAE      : {mae:>10,.2f}  ({metrics["MAE_pct"]:.2f}%)')
    print(f'     RMSE     : {rmse:>10,.2f}  ({metrics["RMSE_pct"]:.2f}%)')
    print(f'     Dir.Acc  : {da:.2f}%')

    return {
        'dates_te': dates_te,
        'y_true':   y_true,
        'y_pred':   y_pred,
        'model':    m,
        'metrics':  metrics,
    }

print('NeuralProphet model functions defined OK')

NeuralProphet model functions defined OK


## Cell 10 — Comparison Plot & Summary

In [ ]:
# ============================================================
#  CELL 10 -- COMPARISON PLOT + SUMMARY TABLE
# ============================================================
def plot_np_comparison(df_weekly_price, res_bl, res_hyb, test_size):
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle('BBCA -- NeuralProphet Price-Only vs Hybrid (MIDAS Weekly)\n'
                 '(Weekly MIDAS Price  vs  Weekly Price + R-MIDAS Fundamentals)',
                 fontsize=13, fontweight='bold', y=0.99)
    gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.5, wspace=0.32)

    cutoff    = df_weekly_price['ds'].iloc[-test_size]
    train_act = df_weekly_price[df_weekly_price['ds'] <  cutoff]
    test_act  = df_weekly_price[df_weekly_price['ds'] >= cutoff]

    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(train_act['ds'], train_act['y'], color='lightsteelblue', lw=0.9, label='Training Period')
    ax1.plot(test_act['ds'],  test_act['y'],  'g-', lw=1.3, label='Actual (Test)')
    ax1.plot(pd.to_datetime(res_bl['dates_te']),  res_bl['y_pred'],  'b--', lw=1.2, label='Price-Only')
    ax1.plot(pd.to_datetime(res_hyb['dates_te']), res_hyb['y_pred'], color='darkorange', linestyle='--', lw=1.2, label='Hybrid')
    ax1.set_title('Full Weekly History + Test Period'); ax1.set_xlabel('Week'); ax1.set_ylabel('Close (IDR)')
    ax1.legend(loc='upper left'); ax1.grid(True, alpha=0.3)

    ax2 = fig.add_subplot(gs[1, :])
    m_bl, m_hyb = res_bl['metrics'], res_hyb['metrics']
    ax2.plot(pd.to_datetime(res_bl['dates_te']), res_bl['y_true'], 'g.-', lw=1.2, label='Actual')
    ax2.plot(pd.to_datetime(res_bl['dates_te']),  res_bl['y_pred'],  'b--', lw=1.1,
             label=f"Price-Only  MAE={m_bl['MAE']:,.0f} ({m_bl['MAE_pct']:.1f}%)")
    ax2.plot(pd.to_datetime(res_hyb['dates_te']), res_hyb['y_pred'], color='darkorange', linestyle='--', lw=1.1,
             label=f"Hybrid      MAE={m_hyb['MAE']:,.0f} ({m_hyb['MAE_pct']:.1f}%)")
    ax2.set_title(f'Zoomed -- Last {test_size} Weeks (Test Set)'); ax2.set_xlabel('Week'); ax2.set_ylabel('Close (IDR)')
    ax2.legend(loc='upper left'); ax2.grid(True, alpha=0.3)

    ax3 = fig.add_subplot(gs[2, 0])
    names = ['Price-Only', 'Hybrid']
    x, w = np.arange(2), 0.35
    mae_v  = [m_bl['MAE_pct'],  m_hyb['MAE_pct']]
    rmse_v = [m_bl['RMSE_pct'], m_hyb['RMSE_pct']]
    b1 = ax3.bar(x - w/2, mae_v,  w, color=['steelblue','darkorange'], alpha=0.9,  label='MAE (%)')
    b2 = ax3.bar(x + w/2, rmse_v, w, color=['steelblue','darkorange'], alpha=0.55, label='RMSE (%)')
    ax3.bar_label(b1, fmt='%.2f%%', padding=2, fontsize=9)
    ax3.bar_label(b2, fmt='%.2f%%', padding=2, fontsize=9)
    ax3.set_title('Regression Error (lower is better)'); ax3.set_xticks(x); ax3.set_xticklabels(names)
    ax3.set_ylabel('% of mean price'); ax3.legend(); ax3.grid(True, alpha=0.3, axis='y')

    ax4 = fig.add_subplot(gs[2, 1])
    da_v = [m_bl['Dir_Acc'], m_hyb['Dir_Acc']]
    bars = ax4.bar(names, da_v, color=['steelblue','darkorange'], alpha=0.88, width=0.4)
    ax4.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=10)
    ax4.axhline(50, linestyle='--', color='grey', lw=0.8, label='Random (50%)')
    ax4.set_title('Directional Accuracy (higher is better)'); ax4.set_ylabel('Accuracy (%)'); ax4.set_ylim(0, 100)
    ax4.legend(); ax4.grid(True, alpha=0.3, axis='y')

    plt.savefig('np_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('[Saved] np_comparison.png')


def print_np_summary(res_bl, res_hyb):
    m_bl, m_hyb = res_bl['metrics'], res_hyb['metrics']
    dm  = (m_bl['MAE']  - m_hyb['MAE'])  / m_bl['MAE']  * 100
    dr  = (m_bl['RMSE'] - m_hyb['RMSE']) / m_bl['RMSE'] * 100
    dda = m_hyb['Dir_Acc'] - m_bl['Dir_Acc']

    print('\n' + '='*62)
    print('  NEURALPROPHET FINAL RESULTS')
    print('='*62)
    print(f"  {'Metric':<26} {'Price-Only':>16} {'Hybrid':>16}")
    print(f"  {'-'*58}")
    for label, k, fmt in [
        ('MAE (IDR)',     'MAE',      '{:>15,.2f}'),
        ('MAE (%)',       'MAE_pct',  '{:>14.2f}%'),
        ('RMSE (IDR)',    'RMSE',     '{:>15,.2f}'),
        ('RMSE (%)',      'RMSE_pct', '{:>14.2f}%'),
        ('Dir. Accuracy', 'Dir_Acc',  '{:>14.2f}%'),
    ]:
        print(f"  {label:<26}" + fmt.format(m_bl[k]) + fmt.format(m_hyb[k]))
    print(f"  {'-'*58}")
    print(f"  Hybrid vs Price-Only  ->  MAE {dm:+.2f}%   RMSE {dr:+.2f}%   Dir.Acc {dda:+.2f}pp")
    print('='*62)

print('Plotting + summary functions defined OK')

Plotting + summary functions defined OK


: 

## Cell 11 — Main (runs the full MIDAS + NeuralProphet pipeline)

In [ ]:
# ============================================================
#  CELL 11 -- MAIN
# ============================================================
def main():
    # 1. Load raw data
    df_price_daily = load_price_data(PRICE_PATH)
    fund_m         = load_monthly_pdf_data(PDF_DIR, PARSED_CACHE)
    avail_cols     = [c for c in FUNDAMENTAL_COLS if c in fund_m.columns]
    print(f'[Config]  Fundamental features: {avail_cols}')

    # 2. MIDAS -- daily price -> weekly price (Section III-B, High-Frequency Aggregation)
    df_price_weekly = midas_weekly_price(df_price_daily)

    # 3. R-MIDAS -- monthly fundamentals -> weekly grid (Low-Frequency Disaggregation)
    weekly_fund = rmidas_weekly_fundamentals(
        df_price_weekly['ds'].values, fund_m, avail_cols)

    # 4. Merge price + fundamentals onto one weekly frame
    df_full = df_price_weekly.merge(weekly_fund, on='ds', how='inner')
    df_full = df_full.dropna(subset=avail_cols).reset_index(drop=True)
    print(f'[Merge]  {len(df_full)} aligned weekly rows  '
          f'({df_full["ds"].min().date()} -> {df_full["ds"].max().date()})')

    # 5. Train-only z-score normalisation (Section III-B, Feature Normalization)
    cutoff     = df_full['ds'].iloc[-TEST_SIZE]
    train_only = df_full[df_full['ds'] < cutoff]
    norm_cols  = ['y'] + avail_cols
    norm_stats = fit_normaliser(train_only, norm_cols)
    df_full_n  = apply_normaliser(df_full, norm_stats)

    print(f'\n[Split]  Cutoff: {cutoff.date()}'
          f'  |  Train: {(df_full["ds"] < cutoff).sum():,} weeks'
          f'  |  Test: {TEST_SIZE} weeks')

    # 6. Model 1 -- Price-Only (Section III-C)
    print('\n[Step 6]  Training Price-Only NeuralProphet ...')
    res_price_only = train_neuralprophet(
        df_full_n, fund_cols=None, cutoff=cutoff,
        y_stats=norm_stats, name='Price-Only')

    # 7. Model 2 -- Hybrid (Section III-C)
    print('\n[Step 7]  Training Hybrid NeuralProphet ...')
    res_hybrid = train_neuralprophet(
        df_full_n, fund_cols=avail_cols, cutoff=cutoff,
        y_stats=norm_stats, name='Hybrid')

    # 8. Plots + summary (Section III-D, Model Evaluation)
    print('\n[Step 8]  Generating comparison plot ...')
    plot_np_comparison(df_full, res_price_only, res_hybrid, TEST_SIZE)
    print_np_summary(res_price_only, res_hybrid)

    print('\n  Output files:')
    for f in ['np_comparison.png']:
        status = 'OK' if os.path.exists(f) else 'MISSING'
        print(f'    [{status}] {f}')

    return res_price_only, res_hybrid


res_price_only, res_hybrid = main()

[Price]  2015-01-02 -> 2025-12-30  |  2,659 trading days
[Monthly]  Cache: 106 months loaded.
[Monthly]  Converting YTD cumulative -> standalone monthly ...
[Monthly]  105 months ready  (2015-03-31 -> 2025-11-30)
[Monthly]  Features: ['net_income', 'revenue', 'net_profit_margin', 'ni_growth_yoy', 'rev_growth_yoy']
[Config]  Fundamental features: ['net_income', 'revenue', 'net_profit_margin', 'ni_growth_yoy', 'rev_growth_yoy']
[MIDAS]  WARNING: 7 week(s) are not exactly 7 days apart (likely a fully non-trading ISO week, e.g. a holiday week).
[MIDAS]  Weekly price series: 568 weeks  (2014-12-29 -> 2025-12-29)
[R-MIDAS]  Weekly fundamentals: 548 weeks  (publication lag = 45 days, lookback = 3 reports)
[Merge]  548 aligned weekly rows  (2015-05-18 -> 2025-12-29)

[Split]  Cutoff: 2023-09-04  |  Train: 428 weeks  |  Test: 120 weeks

[Step 6]  Training Price-Only NeuralProphet ...

  [Price-Only]  train=428 weeks  test=120 weeks  features=0
Training: |          | 0/? [00:00<?, ?it/s]

Finding best initial lr: 100%|██████████| 218/218 [00:00<00:00, 224.79it/s]

## Appendix A — Debug PDF

In [ ]:
# Change to any PDF path to debug
DEBUG_PDF = 'monthly_reports/BBCA_2015_08.pdf'

with pdfplumber.open(DEBUG_PDF) as pdf:
    raw_text = '\n'.join(p.extract_text() or '' for p in pdf.pages)

annual = _is_annual_format(raw_text)
print(f'Format: {"ANNUAL (December)" if annual else "MONTHLY"}')
print('\n=== RAW TEXT (first 2000 chars) ===')
print(raw_text[:2000])

print('\n=== EXTRACTED VALUES ===')
vals = _extract_fields(raw_text, _PAT_MONTHLY)
if annual:
    vals.update(_extract_fields(raw_text, _PAT_RATIOS))

for k, v in vals.items():
    if k == 'is_annual':
        continue
    try:
        vf = float(v)
        if not np.isnan(vf):
            print(f'  {k:<28}: {vf:>15,.0f} M IDR  ({vf/1000:>10,.1f} B IDR)')
        else:
            print(f'  {k:<28}: NOT FOUND')
    except Exception:
        print(f'  {k:<28}: {v}')

# Expected for BBCA_2015_08.pdf:
# interest_income : 28,772,308 M IDR
# net_profit      : 11,724,925 M IDR
# total_assets    : 569,985,436 M IDR
# total_loans     : 358,988,534 M IDR
# interest_expense:  7,356,116 M IDR

Format: MONTHLY

=== RAW TEXT (first 2000 chars) ===
PT BANK CENTRAL ASIA Tbk
STATEMENTS OF FINANCIAL POSITION
As of August 31, 2015
(In millions of Rupiah)
BANK
No. ACCOUNTS Unaudited
August 31, 2015
ASSETS
1. Cash 13,815,178
2. Placement to Bank Indonesia 68,082,675
3. Interbank placement 13,076,870
4. Spot and derivatives claims 26,646
5. Securities 67,292,481
a. Measured at fair value through profit and loss 1,137,387
b. Available for sale 49,059,656
c. Held to maturity 14,255,063
d. Loan and receivables 2,840,375
6. Securities sold under repurchase agreement
(repo) -
7. Claims on securities bought under reverse
repo 27,354,383
8. Acceptance claims 6,444,849
9. Loans 358,988,534
a. Measured at fair value through profit and loss -
b. Available for sale -
c. Held to maturity -
d. Loan and receivables 358,988,534
10. Sharia Financing -
11. Equity investment 1,848,475
12. Impairment on financial assets -/- (8,187,231)
a. Securities (737,810)
b. Loans (7,209,370)
c. Others (240,051)
13.

## Appendix B — YTD check

In [ ]:
fund_check = load_monthly_pdf_data(PDF_DIR, PARSED_CACHE)
print(fund_check[['ds', 'net_profit_margin', 'ni_growth_yoy', 'rev_growth_yoy']]
      .head(36).to_string())

plt.figure(figsize=(12, 3))
plt.plot(fund_check['ds'], fund_check['net_profit_margin'],
         marker='o', markersize=3, lw=1, color='steelblue')
plt.title('Net Profit Margin after YTD fix -- should be smooth (no sawtooth)')
plt.ylim(0, 1); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ytd_check.png', dpi=120, bbox_inches='tight')
plt.close()
print('[Saved] ytd_check.png')

[Monthly]  Cache: 106 months loaded.
[Monthly]  Converting YTD cumulative -> standalone monthly ...
[Monthly]  105 months ready  (2015-03-31 -> 2025-11-30)
[Monthly]  Features: ['net_income', 'revenue', 'net_profit_margin', 'ni_growth_yoy', 'rev_growth_yoy']
           ds  net_profit_margin  ni_growth_yoy  rev_growth_yoy
0  2015-03-31           0.073269       0.000000        0.000000
1  2015-04-30           1.000000       0.000000        0.000000
2  2015-05-31           0.446147       0.000000        0.000000
3  2015-06-30           0.396189       0.000000        0.000000
4  2015-07-31           0.486998       0.000000        0.000000
5  2015-08-31           0.464839       0.000000        0.000000
6  2015-09-30           0.330567       0.000000        0.000000
7  2015-10-31           0.475742       0.000000        0.000000
8  2015-11-30           0.527697       0.000000        0.000000
9  2015-12-31           0.225793       0.000000        0.000000
10 2016-01-31           0.422959     